# Weaver: from disorganized to organized complexity

This is a deliberately simple **desire-path** simulation.

We use exactly the same:

- people,
- space,
- destinations,
- movement possibilities.

We compare only two situations:

1. **No memory / no feedback:** walkers choose independently among reasonable steps toward their destination.
2. **Memory + feedback:** walkers can see which places were used before and tend to follow them.

The question is simple:

> What changes when past actions begin to organize future actions?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# World
WIDTH = 41
HEIGHT = 31

# Three destinations
buildings = [
    (3, 5),
    (3, 25),
    (37, 15)
]


## The model

Each walker repeatedly travels from one building to another.

At every step, the walker considers only neighboring cells that take it **closer to its destination**.

- With **memory OFF**, it chooses randomly among those cells.
- With **memory ON**, it usually chooses the most popular available cell.

Every visit increases a cell's `popularity`.

That is the only difference between the two simulations.


In [ ]:
def closer_neighbors(x, y, gx, gy):
    current_distance = (x - gx)**2 + (y - gy)**2

    candidates = []

    for dx, dy in [
        (1, 0), (-1, 0), (0, 1), (0, -1),
        (1, 1), (1, -1), (-1, 1), (-1, -1)
    ]:
        nx = x + dx
        ny = y + dy

        if 0 <= nx < WIDTH and 0 <= ny < HEIGHT:
            new_distance = (nx - gx)**2 + (ny - gy)**2

            if new_distance < current_distance:
                candidates.append((nx, ny))

    return candidates


In [ ]:
def run_paths(memory=False, seed=123,
              n_walkers=80,
              total_steps=20000,
              follow_probability=0.85):

    rng = np.random.default_rng(seed)

    popularity = np.zeros((HEIGHT, WIDTH), dtype=int)

    positions = []
    goals = []

    # Every walker starts at one building
    # and receives another building as its destination.
    for _ in range(n_walkers):
        start, goal = rng.choice(
            len(buildings),
            size=2,
            replace=False
        )

        positions.append(buildings[start])
        goals.append(buildings[goal])

    steps = 0

    while steps < total_steps:

        for i in range(n_walkers):

            x, y = positions[i]
            gx, gy = goals[i]

            # Arrived: choose another building
            if (x, y) == (gx, gy):
                current = buildings.index((gx, gy))

                possible_goals = [
                    j for j in range(len(buildings))
                    if j != current
                ]

                new_goal = rng.choice(possible_goals)
                goals[i] = buildings[new_goal]

                continue

            candidates = closer_neighbors(x, y, gx, gy)

            # MEMORY OFF:
            # choose independently among reasonable steps.
            if not memory:
                nx, ny = candidates[
                    rng.integers(len(candidates))
                ]

            # MEMORY ON:
            # usually follow the most-used available cell.
            else:
                if rng.random() < follow_probability:

                    values = [
                        popularity[cy, cx]
                        for cx, cy in candidates
                    ]

                    best_value = max(values)

                    best_cells = [
                        cell for cell, value
                        in zip(candidates, values)
                        if value == best_value
                    ]

                    nx, ny = best_cells[
                        rng.integers(len(best_cells))
                    ]

                else:
                    nx, ny = candidates[
                        rng.integers(len(candidates))
                    ]

            positions[i] = (nx, ny)

            popularity[ny, nx] += 1

            steps += 1

            if steps >= total_steps:
                break

    return popularity


## Case A — No memory

Walkers have the same destinations and the same possible movements, but each movement decision is independent of previous walkers.

Past movement leaves no information that later walkers use.


In [ ]:
no_memory = run_paths(
    memory=False,
    seed=123
)

plt.figure(figsize=(8, 6))
plt.imshow(no_memory, origin="lower")

for x, y in buildings:
    plt.scatter(x, y, marker="s", s=80)

plt.title("No memory: cumulative foot traffic")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## Case B — Memory and feedback

Now walkers can respond to the history of the system.

Frequently used places become more attractive to later walkers.

So:

**movement → popularity → later movement → more popularity**


In [ ]:
with_memory = run_paths(
    memory=True,
    seed=123
)

plt.figure(figsize=(8, 6))
plt.imshow(with_memory, origin="lower")

for x, y in buildings:
    plt.scatter(x, y, marker="s", s=80)

plt.title("Memory + feedback: cumulative foot traffic")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## A very simple comparison

If movement becomes organized around a smaller number of cells, a greater share of all footsteps should be concentrated in the most-used parts of the landscape.

We measure the share of all footsteps occurring in the **top 5% most-used cells**.


In [ ]:
def top_share(popularity, fraction=0.05):
    values = np.sort(popularity.ravel())[::-1]

    n_top = max(
        1,
        int(len(values) * fraction)
    )

    return values[:n_top].sum() / values.sum()


print(
    "No memory:",
    round(top_share(no_memory), 3)
)

print(
    "Memory + feedback:",
    round(top_share(with_memory), 3)
)


# What should we notice?

In both simulations:

- there are the same walkers;
- they move in the same world;
- they travel among the same destinations;
- they have the same possible steps.

The difference is **memory and feedback**.

### Without memory

A walker's action does not organize the choices of later walkers.

Many individual movements occur, but their histories do not become part of the system.

### With memory

Past movements change the information available to later walkers.

Repeated use reinforces some routes:

**past behavior → structure → future behavior**

This provides a simple visual analogy for Weaver's distinction.

> Organized complexity is not simply about having many elements.  
> The relationships among actions — including feedback through structures created by earlier actions — can become essential for understanding the behavior of the whole.

### Important

This is a teaching illustration, not a claim that every system with feedback is automatically a complete example of Weaver's organized complexity.
